In [ ]:
# import folium


# map = folium.Map(
#     zoom_start=12,
# )

# folium.Rectangle(((46, 7), (49, 11))).add_to(map)
# folium.GeoJson(cov).add_to(map)
# map

In [ ]:
from theia.types import Point
from theia.plotting import plot_profile


p1 = Point(lat=46.55, lon=8.0, alt=4000)
p2 = Point(lat=46.55, lon=8.6, alt=1500)


plot_profile(p1, p2)

# Load terrain

In [ ]:
from theia.terrain_fast_los import HbvTree, FastSrtmModel, SrtmTerrainModel
from theia.coordinates import CoordinateTransformations
import numpy as np

srtm = SrtmTerrainModel()
tree = HbvTree.load("tree_lat46:49_lon7:11_subsamplestride2.zip")

In [ ]:
fast = FastSrtmModel(tree=tree, srtm_model=srtm, t_min=35)

## Benchmark terrain

### LOS

In [ ]:
# warmup
srtm.has_line_of_sight(p1, p2), fast.has_line_of_sight(p1, p2)

In [ ]:
%%timeit
srtm.has_line_of_sight(p1, p2)

In [ ]:
%%timeit
fast.has_line_of_sight(p1, p2)

### Coverage calc

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range
from theia.test_data import build_flores_monostatic_radar
from theia.types import Point
from theia.terrain import SrtmTerrainModel


rad = build_flores_monostatic_radar(
    Point(
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    1,
    2,
    1,
)
rad_max = calculate_maximum_monostatic_range(rad, 1.0)

In [ ]:
from theia.distance import burstvincentydistance


p = burstvincentydistance(
    (rad.receiver.point.lat, rad.receiver.point.lon),
    dist_m=1000,
    brng=0,
    alt=1000.0,
)
# fast.has_line_of_sight()
print(rad.receiver.point)
print(p)
fast.has_line_of_sight(
    Point(
        lat=rad.receiver.point.lat,
        lon=rad.receiver.point.lon,
        alt=rad.receiver.alt,
    ),
    p,
)

In [ ]:
cov1 = calculate_coverage(
    fast,
    rad.receiver.point,
    rad_max,
    1000.0,
    dist_res=30,
    d_theta=2,
    n_workers=1,
)

In [ ]:
cov2 = calculate_coverage(
    SrtmTerrainModel(),
    rad.receiver.point,
    rad_max,
    1000.0,
    dist_res=30,
    d_theta=2,
    n_workers=5,
)

In [ ]:
cov1

In [ ]:
cov2

In [ ]:
p = Point(lat=48.2390, lon=8.4609, alt=1000)

In [ ]:
fig, ax = plot_profile(rad.receiver.point, p, "Rx", "Tgt")
ax.set_xlim([0, 0.25])
ax.set_ylim([800, 900])

In [ ]:
import folium

map = folium.Map(location=(47.5755, 8.7152), zoom_start=8)
folium.GeoJson(cov1, color="red").add_to(map)
folium.GeoJson(cov2).add_to(map)
folium.LatLngPopup().add_to(map)
map

In [ ]:
rng = np.random.default_rng(seed=3097599)

start_points = [fast.sample_location(rng, 46, 49, 7, 11) for _ in range(50_000)]
stop_points = [fast.sample_location(rng, 46, 49, 7, 11) for _ in range(50_000)]

In [ ]:
import tracemalloc


for p1, p2 in zip(start_points, stop_points, strict=True):
    fast.has_line_of_sight(p1, p2)

In [ ]:
srtm.has_line_of_sight(p1, p2)

In [ ]:
fast.has_line_of_sight(p1, p2)

In [ ]:
%%timeit
srtm.has_line_of_sight(p1, p2)

In [ ]:
%%timeit
fast.has_line_of_sight(p1, p2)

In [ ]:
import cProfile


profiler = cProfile.Profile()
profiler.enable()

for _ in range(10_000):
    fast.has_line_of_sight(p1, p2)

profiler.disable()
profiler.dump_stats("profile_los_bbox.prof")

In [ ]:
from theia.terrain import SrtmTerrainModel
from theia.types import Point

ref_terrain = SrtmTerrainModel()


p1 = Point(lat=46.5, lon=8.2, alt=5000)  # well above terrain near Grimsel
p2 = Point(lat=46.8, lon=8.7, alt=5000)  # well above terrain near Gotthard
# Expected: no intersection

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST
from theia.radar_equation import calculate_maximum_monostatic_range
from theia.test_data import build_flores_monostatic_radar
from theia.types import Point
from theia.coverage import calculate_coverage
from theia.terrain import SrtmTerrainModel


rad = build_flores_monostatic_radar(
    Point(
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    1,
    2,
    1,
)
rad_max = calculate_maximum_monostatic_range(rad, 1.0)

In [ ]:
calculate_coverage(
    SrtmTerrainModel(),
    rad.receiver.point,
    rad_max,
    1000.0,
    d_theta=2,
);

In [ ]:
%%timeit
calculate_coverage(
    srtm,
    rad.receiver.point,
    rad_max,
    1000.0,
    d_theta=2,
)

In [ ]:
fast = FastSrtmModel(node=head, srtm_model=srtm)

In [ ]:
calculate_coverage(
    fast,
    rad.receiver.point,
    rad_max,
    1000.0,
    d_theta=2,
);

In [ ]:
# Idea:
# Place PET receivers so that they have as much line of sight to the
# target trajectory as possible. That means:
# - Sample target waypoints.
# - For each waypoint: Calculate visible points on lat-lon-terrain-grid (binary mask)
# - Sum up the masks for each waypoint.
# - Place sensors at maxima.

# TODO:
# Replace sidc code by "TargetCategory", which has an "archetype" and a SIDC code.
# Open for extension in the future.

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST


lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]

In [ ]:
from theia.terrain import elevationAt


elevationAt(lat, lon)

In [ ]:
import abc
import math

from theia.config import ELEVATION_DATA_DIR
from theia.terrain import interpolate_elevation_tile, load_hgt_file
from theia.types import Point


model = SrtmTerrainModel()

In [ ]:
%%timeit
alt = elevationAt(lat, lon)

In [ ]:
%%timeit
alt = model.elevationAt(lat, lon)

In [ ]:
import numba
import numpy as np


@numba.njit(cache=True, inline="always")
def _norm3(v):
    return np.sqrt(v[0] * v[0] + v[1] * v[1] + v[2] * v[2])


@numba.njit(cache=True, inline="always")
def _orthonormal_pair(e):
    """Two unit vectors perpendicular to unit vector e."""
    if abs(e[0]) < 0.9:
        t = np.array([1.0, 0.0, 0.0])
    else:
        t = np.array([0.0, 1.0, 0.0])
    # e2 = t − (t·e)e  normalised
    dot = t[0] * e[0] + t[1] * e[1] + t[2] * e[2]
    e2 = np.array([t[0] - dot * e[0], t[1] - dot * e[1], t[2] - dot * e[2]])
    e2 /= _norm3(e2)
    # e3 = e × e2
    e3 = np.array(
        [
            e[1] * e2[2] - e[2] * e2[1],
            e[2] * e2[0] - e[0] * e2[2],
            e[0] * e2[1] - e[1] * e2[0],
        ]
    )
    return e2, e3


@numba.njit(cache=True)
def _sample_on_ellipsoid(T, R, brange):
    """
    Sample one point uniformly on the prolate-spheroid surface
    defined by foci T, R and bistatic range brange.
    """
    a = 0.5 * brange
    fv = R - T
    c = 0.5 * _norm3(fv)  # focal half-distance

    if c >= a:  # degenerate (invalid range)
        return 0.5 * (T + R)

    b2 = a * a - c * c
    b = np.sqrt(b2)
    a2 = a * a

    e = fv / (2.0 * c)  # unit major-axis vector
    e2, e3 = _orthonormal_pair(e)
    ctr = 0.5 * (T + R)

    # Rejection sampling for surface-uniform (θ, φ)
    while True:
        cos_t = 2.0 * np.random.random() - 1.0  # uniform-on-sphere proposal
        sin_t = np.sqrt(max(0.0, 1.0 - cos_t * cos_t))
        phi = 2.0 * np.pi * np.random.random()

        # Accept prob = sqrt(b²cos²θ + a²sin²θ) / a
        acc = np.sqrt(b2 * cos_t * cos_t + a2 * sin_t * sin_t) / a
        if np.random.random() <= acc:
            cp = np.cos(phi)
            sp = np.sin(phi)
            x = ctr + (a * cos_t) * e + (b * sin_t * cp) * e2 + (b * sin_t * sp) * e3
            return x

In [ ]:
%%timeit
_sample_on_ellipsoid(np.array((0, 0, 0)), np.array((4000, 0, 0)), 150_000)

In [ ]:
from theia.ellipsoid import Ellipsoid


ellipsoid = Ellipsoid((0, 0, 0), (4000, 0, 0), 150_000)

In [ ]:
%%timeit
points = ellipsoid.sample_surface(1, 1)

In [ ]:
# import datetime

# from theia.detection.pet import suggest_pet_receiver_locations
# from theia.grids import LatLonTerrainGrid
# from theia.test_data import build_single_target_from_Bodensee

# trajectory = build_single_target_from_Bodensee(target_id=0)
# targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))
# target_positions = [t.point for t in targets]

# grid = LatLonTerrainGrid(
#     lat_start=46.78125,
#     lat_stop=48.12577,
#     lon_start=7.17484,
#     lon_stop=9.49275,
#     lat_res=0.01,
#     lon_res=0.01,
# )

# best_points = suggest_pet_receiver_locations(grid, target_positions)

In [ ]:
from theia.config import SIDC_RED_FIXED_WING
from theia.simulation.controllers.waypoint_target_controller import (
    WaypointTargetController,
)
from theia.terrain import elevationAt
from theia.test_data import build_fighter_jet_radar, build_single_target_from_Bodensee
from theia.types import Point, Receiver


def build_pet_only_scenario():
    pet_receiver_positions = [
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
    ]

    pet_receivers: list[Receiver] = []
    for i, pos in enumerate(pet_receiver_positions):
        rx = build_fighter_jet_radar(0, 0, 0).receiver.model_copy(deep=True)
        rx.point = pos
        rx.id = i
        pet_receivers.append(rx)

    trajectory = build_single_target_from_Bodensee(target_id=0)
    red_controller = WaypointTargetController.from_trajectory(
        trajectory=trajectory,
        name="Emitting target",
        sidc=SIDC_RED_FIXED_WING,
    )

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
for i, point in enumerate(best_points):
    folium.Marker(
        (point[0], point[1]),
        tooltip=f"Point index = {i}<br />Lat={point[0]:.4f} °<br />Lon={point[1]:.4f} °",
    ).add_to(map)
map

In [ ]:
plt.imshow(n_los)

In [ ]:
%%timeit
los = has_line_of_sight(
    target_pos,
    Point(lat=point[0], lon=point[1], alt=point[2]),
    60,
)

In [ ]:
from theia.coverage import calculate_coverage


calculate_coverage

In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("log.json")

In [ ]:
loader.red_monostatic_radar_detections

In [ ]:
import datetime

from theia.coordinates import POSITIONS_OF_INTEREST
from theia.test_data import (
    build_fighter_jet_radar,
    build_flores_monostatic_radar,
    build_single_target_from_Bodensee,
)
from theia.types import Point


trajectory = build_single_target_from_Bodensee()
target = trajectory(
    datetime.datetime(
        year=2026,
        month=4,
        day=29,
        hour=0,
        minute=0,
        second=58,
    )
)
red_sensor = build_fighter_jet_radar(0, 0, 0)
red_sensor.receiver.point = target.point
red_sensor.transmitter.point = target.point
blue_sensor = build_flores_monostatic_radar(
    Point(
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    1,
    1,
    1,
)

In [ ]:
from theia.line_of_sight import has_line_of_sight


has_line_of_sight(red_sensor.receiver.point, blue_sensor.receiver.point, 30)

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range


r_max_red = calculate_maximum_monostatic_range(red_sensor, 1.0)
r_max_blue = calculate_maximum_monostatic_range(blue_sensor, 1.0)

In [ ]:
from theia.coverage import calculate_coverage


coverage_red = calculate_coverage(
    red_sensor.receiver.point, r_max_red, blue_sensor.receiver.alt, d_theta=2
)
coverage_blue = calculate_coverage(
    blue_sensor.receiver.point, r_max_blue, 1000.0, d_theta=2
)

In [ ]:
from theia.mapping import RadarMap


RadarMap(
    sensors={
        "RED": red_sensor,
        "BLUE": blue_sensor,
    },
    polygons={
        "Coverage RED": coverage_red,
        # "Coverage BLUE": coverage_blue,
    },
).to_map()

In [ ]:
from theia.plotting import plot_profile


plot_profile(
    target.point,
    blue_sensor.receiver.point,
    "RED sensor",
    "BLUE sensor (target)",
)

In [ ]:
radar = loader.red_monostatic_radars[0]

In [ ]:
%%timeit
res = calculate_coverage(radar.receiver.point, r_max, 0.0)